# Getting started: weather records and chill metrics

This notebook starts with a small synthetic daily weather record, checks and fixes the
date/temperature structure, converts daily extremes to hourly temperatures, and calculates
standard chill and heat metrics. The data in `examples/data/` are synthetic and deterministic.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from chillPy import (
    check_temperature_record,
    chilling,
    daily_chill,
    fix_weather,
    make_chill_plot,
    stack_hourly_temps,
)

DATA = Path("examples/data")
weather = pd.read_csv(DATA / "synthetic_daily_weather.csv")
weather.head()

Create a small imperfection so the completion and interpolation workflow has something to repair.

In [ ]:
demo_weather = weather.query("2018 <= Year <= 2021").copy()
demo_weather = demo_weather.drop(demo_weather.query("Year == 2019 and Month == 2 and Day == 10").index)
demo_weather.loc[
    (demo_weather["Year"] == 2020) & (demo_weather["Month"] == 1) & (demo_weather["Day"] == 15),
    "Tmin",
] = pd.NA

before = check_temperature_record(demo_weather)
fixed = fix_weather(demo_weather, start_year=2018, end_year=2021, end_at_present=False)
after = check_temperature_record(fixed["weather"])

{
    "before_valid": before["valid"],
    "before_error": before["error"],
    "after_valid": after["valid"],
    "fixed_rows": len(fixed["weather"]),
}

Convert fixed daily records to hourly temperatures and summarize seasonal chill/heat accumulation.

In [ ]:
hourly = stack_hourly_temps(fixed["weather"], latitude=42.0)
chill_summary = chilling(hourly, start_jday=305, end_jday=60)
chill_summary[["End_year", "Chilling_Hours", "Utah_Model", "Chill_portions", "GDH", "Perc_complete"]]

Daily chill is useful for plotting how metrics accumulate through the season.

In [ ]:
daily = daily_chill(hourly, running_mean=3)
plot = make_chill_plot(
    daily,
    metrics=["Chilling_Hours", "Chill_Portions", "GDH"],
    startdate=305,
    enddate=80,
    cumulative=True,
    focusyears=[2020, 2021],
)
plot["figure"]